# Chapter 6 Walkthrough: HCP and Account Targeting

This notebook builds the chapter artifact from the Chapter 3 synthetic source tables and the Chapter 5 journey output. The analysis date is December 31, 2024.


In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
while not (ROOT / "ch06_hcp").exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("Run this notebook inside the repository.")
    ROOT = ROOT.parent

SCRIPT_DIR = ROOT / "ch06_hcp" / "scripts"
sys.path.insert(0, str(SCRIPT_DIR))

from run_analysis import load_inputs, write_outputs
from targeting import (
    apply_account_policy,
    build_account_features,
    build_call_plan,
    build_decile_summary,
    build_gate_summary,
    build_hcp_actions,
    build_hcp_features,
    build_territory_summary,
    compare_naive_and_gated,
)


## 1. Load the source tables


In [2]:
inputs = load_inputs(ROOT)
inventory = pd.DataFrame(
    {"table": inputs.keys(), "rows": [len(frame) for frame in inputs.values()]}
)
display(inventory)


,table,rows
0,journeys,6393
1,medical_claims,49882
2,hcp_roster,281
3,providers,666
4,accounts,250
5,crm,3359


The patient journey is the analysis cohort. Medical claims provide the diagnosis-index rendering HCP. The field roster links that HCP to an account, while CRM provides recency and current permission.


## 2. Build HCP evidence


In [3]:
hcp_features, patient_hcp = build_hcp_features(inputs)
coverage = patient_hcp.patient_id.nunique() / inputs["journeys"].patient_id.nunique()
print(f"Target-roster patients: {patient_hcp.patient_id.nunique():,}")
print(f"Target-roster coverage: {coverage:.1%}")
print(f"HCPs: {hcp_features.npi.nunique():,}")

display(
    hcp_features.nlargest(10, "cohort_patients")[[
        "npi", "account_name", "cohort_patients", "opportunity_patients",
        "recent_contacts", "consent_status"
    ]]
)


Target-roster patients: 2,749
Target-roster coverage: 43.0%
HCPs: 281


,npi,account_name,cohort_patients,opportunity_patients,recent_contacts,consent_status
167,9000000430,Michigan Care 189,36,20,0,Allowed
220,9000000555,Florida Care 064,36,21,1,Allowed
185,9000000469,Illinois Care 121,34,20,1,Opt-out
63,9000000162,Arizona Care 062,33,13,2,Opt-out
175,9000000447,New York Care 216,32,17,2,Opt-out
102,9000000249,Florida Care 147,29,13,3,Allowed
138,9000000349,Ohio Care 184,29,11,0,Allowed
8,9000000026,Washington Care 226,28,10,1,Allowed
129,9000000331,Michigan Care 154,28,16,2,Allowed
212,9000000537,Texas Care 079,28,16,0,Opt-out


The target roster covers 43.0% of the journey cohort. The denominator stays visible because the table describes the field roster, not the full provider market.


## 3. Diagnose volume concentration


In [4]:
hcp_deciles, decile_summary = build_decile_summary(hcp_features)
display(
    decile_summary[[
        "volume_decile", "hcps", "cohort_patients", "opportunity_patients",
        "permitted_hcps", "contactable_share"
    ]]
)


,volume_decile,hcps,cohort_patients,opportunity_patients,permitted_hcps,contactable_share
0,1,29,80,41,19,0.655172
1,2,28,124,64,18,0.642857
2,3,28,156,74,22,0.785714
3,4,28,176,85,19,0.678571
4,5,28,210,84,19,0.678571
5,6,28,236,109,20,0.714286
6,7,28,284,136,20,0.714286
7,8,28,340,158,21,0.750000
8,9,28,428,199,19,0.678571
9,10,28,715,347,18,0.642857


Decile 10 carries the largest patient opportunity, while its contactable share is 64.3%. Volume and permission remain separate decision inputs.


## 4. Apply account gates


In [5]:
account_features = build_account_features(hcp_features, inputs["accounts"])
account_targets = apply_account_policy(account_features)
gate_summary = build_gate_summary(account_targets)

display(gate_summary)
display(
    account_targets[[
        "account_name", "cohort_patients", "opportunity_patients",
        "roventra_share", "contactable_hcps", "access_signal_patients",
        "account_action", "action_reason"
    ]].head(20)
)


,stage,accounts
0,Target-roster accounts,165
1,Evidence floor,129
2,Opportunity floor,114
3,No access-review flag,112
4,Contact permission,94
5,Ownership and capacity,94


,account_name,cohort_patients,opportunity_patients,roventra_share,contactable_hcps,access_signal_patients,account_action,action_reason
0,Florida Care 064,36,21,0.789474,1,0,Increase priority,Contactable opportunity and Roventra share bel...
1,Michigan Care 189,36,20,0.761905,1,1,Increase priority,Contactable opportunity and Roventra share bel...
2,Arizona Care 155,38,18,0.714286,1,0,Increase priority,Contactable opportunity and Roventra share bel...
3,Washington Care 233,36,17,0.791667,3,0,Increase priority,Contactable opportunity and Roventra share bel...
4,Michigan Care 057,32,17,0.789474,1,0,Increase priority,Contactable opportunity and Roventra share bel...
5,Washington Care 190,29,16,0.722222,4,0,Increase priority,Contactable opportunity and Roventra share bel...
6,Michigan Care 224,27,15,0.750000,1,0,Increase priority,Contactable opportunity and Roventra share bel...
7,Florida Care 005,24,15,0.529412,3,0,Increase priority,Contactable opportunity and Roventra share bel...
8,Michigan Care 128,22,15,0.700000,1,1,Increase priority,Contactable opportunity and Roventra share bel...
9,Texas Care 045,33,14,0.760000,2,0,Increase priority,Contactable opportunity and Roventra share bel...


The gates leave 94 field-eligible accounts. The action table preserves the access-review and hold queues instead of deleting them.


## 5. Resolve HCP actions and calls


In [6]:
hcp_targets = build_hcp_actions(hcp_deciles, account_targets)
call_plan = build_call_plan(account_targets, hcp_targets)
comparison = compare_naive_and_gated(hcp_targets, call_plan)

display(hcp_targets.hcp_action.value_counts().rename_axis("action").to_frame("hcps"))
display(comparison)
display(call_plan.head(20))


,hcps
action,
Monitor,111
Hold contact,86
Prioritize,56
Maintain,27
Access follow-up,1


,plan,selected_hcps,contact_permitted,opted_out,opportunity_patients,recent_contacts
0,Top 30 by patient volume,30,20,10,361,32
1,Gated near-term plan,30,30,0,267,14


,territory,account_id,account_name,npi,specialty,account_action,hcp_action,recommended_calls,hcp_opportunity_patients,recent_contacts,reason
0,T01,ACC064,Florida Care 064,9000000555,Oncology,Increase priority,Prioritize,1,21,1,Contactable opportunity and Roventra share bel...
1,T01,ACC224,Michigan Care 224,9000000217,Endocrinology,Increase priority,Prioritize,2,15,1,Contactable opportunity and Roventra share bel...
2,T01,ACC056,Pennsylvania Care 056,9000000136,Primary Care,Increase priority,Prioritize,2,7,1,Contactable opportunity and Roventra share bel...
3,T01,ACC024,Arizona Care 024,9000000432,Oncology,Increase priority,Prioritize,1,6,0,Contactable opportunity and Roventra share bel...
4,T01,ACC176,Arizona Care 176,9000000446,Primary Care,Increase priority,Prioritize,1,5,1,Contactable opportunity and Roventra share bel...
5,T01,ACC080,New Jersey Care 080,9000000624,Rheumatology,Increase priority,Prioritize,1,5,1,Contactable opportunity and Roventra share bel...
6,T01,ACC080,New Jersey Care 080,9000000585,Rheumatology,Increase priority,Prioritize,1,4,1,Contactable opportunity and Roventra share bel...
7,T01,ACC048,Washington Care 048,9000000545,Endocrinology,Increase priority,Prioritize,1,3,0,Contactable opportunity and Roventra share bel...
8,T01,ACC176,Arizona Care 176,9000000505,Endocrinology,Increase priority,Prioritize,1,1,0,Contactable opportunity and Roventra share bel...
9,T01,ACC184,Ohio Care 184,9000000349,Oncology,Maintain,Maintain,1,11,0,Contactable opportunity with Roventra share at...


The near-term plan contains 69 permitted HCPs and 82 contacts. The equal-sized comparison shows the tradeoff against a top-30 volume list.


## 6. Audit territory allocation


In [7]:
territory_summary = build_territory_summary(account_targets, call_plan)
display(territory_summary)


,territory,accounts,priority_accounts,opportunity_patients,actionable_opportunity,planned_hcps,recommended_calls,opportunity_share,call_share,allocation_gap
0,T01,24,8,215,170,11,13,0.183190,0.158537,-0.024653
1,T06,21,7,194,170,13,16,0.183190,0.195122,0.011932
2,T07,18,6,170,124,9,12,0.133621,0.146341,0.012721
3,T03,19,3,137,112,4,5,0.120690,0.060976,-0.059714
4,T02,22,7,173,98,10,10,0.105603,0.121951,0.016348
5,T05,22,6,136,94,10,11,0.101293,0.134146,0.032853
6,T08,24,3,165,91,6,6,0.098060,0.073171,-0.024890
7,T04,15,3,107,69,6,9,0.074353,0.109756,0.035403


T03 has 12.1% of field-eligible opportunity and 6.1% of planned calls. This gap belongs in the field-leadership review before release.


## 7. Validate and write the artifact


In [8]:
assert not account_targets.duplicated("account_id").any()
assert hcp_targets.loc[hcp_targets.hcp_action.eq("Prioritize"), "contact_permitted"].all()
assert call_plan.recommended_calls.gt(0).all()
assert set(call_plan.hcp_action) <= {"Prioritize", "Maintain"}
assert account_targets.loc[account_targets.account_action.eq("Access review"), "field_eligible"].eq(False).all()

results = {
    "patient_hcp": patient_hcp,
    "hcp_features": hcp_features,
    "hcp_targets": hcp_targets,
    "hcp_deciles": hcp_deciles,
    "decile_summary": decile_summary,
    "account_features": account_features,
    "account_targets": account_targets,
    "gate_summary": gate_summary,
    "call_plan": call_plan,
    "territory_summary": territory_summary,
    "plan_comparison": comparison,
}
write_outputs(results, ROOT / "ch06_hcp" / "assets" / "generated_outputs")
print("Validation passed and outputs were written.")


Validation passed and outputs were written.


## Conclusion

The transparent policy converts patient evidence into account actions, HCP actions, and capacity-aware activity. The artifact keeps held, monitored, and access-review accounts visible for the owners who need them.
